#SETUP

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
PERPLEXITY_KEY = os.getenv("PERPLEXITY_API_KEY")
os.environ["LANGSMITH_API_KEY"] = ""
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langsmith-academy"
env_path = find_dotenv()
load_dotenv(env_path, override=True)

True

#GROUP TRACES INTO THREADS

In [2]:
import uuid
thread_id = uuid.uuid4()

In [5]:
from langsmith import traceable
import requests
from typing import List
import nest_asyncio
from utils import get_vector_db_retriever
import os

PPLX_API_KEY = os.environ.get("PPLX_API_KEY")
BASE_URL = "https://api.perplexity.ai/chat/completions"

def perplexity_request(messages, model="sonar-pro", temperature=0.0):
    r = requests.post(
        BASE_URL,
        headers={
            "Authorization": f"Bearer {PPLX_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": model,
            "messages": messages,
            "temperature": temperature
        }
    )
    r.raise_for_status()
    return r.json()

nest_asyncio.apply()
retriever = get_vector_db_retriever()

@traceable(run_type="chain")
def retrieve_documents(question: str):
    return retriever.invoke(question)

@traceable(run_type="chain")
def generate_response(question: str, documents):
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)
    rag_system_prompt = """You are an assistant for question-answering tasks. 
    Use the following pieces of retrieved context to answer the latest question in the conversation. 
    If you don't know the answer, just say that you don't know. 
    Use three sentences maximum and keep the answer concise.
    """
    messages = [
        {"role": "system", "content": rag_system_prompt},
        {"role": "user", "content": f"Context: {formatted_docs} \n\n Question: {question}"}
    ]
    return call_openai(messages)

@traceable(run_type="llm")
def call_openai(messages: List[dict], model: str = "sonar-pro", temperature: float = 0.0):
    return perplexity_request(messages, model=model, temperature=temperature)

@traceable(run_type="chain")
def langsmith_rag(question: str):
    documents = retrieve_documents(question)
    response = generate_response(question, documents)
    return response["choices"][0]["message"]["content"]


In [7]:
question = "How do I calculate volatility?"
ai_answer = langsmith_rag(question, langsmith_extra={"volatility": {"thread_id": thread_id}})
print(ai_answer)

To calculate **volatility**, compute the **standard deviation** of the asset’s returns over your chosen period, which measures how much returns deviate from their average[2][6][8]. For **annualized volatility**, multiply the standard deviation of daily returns by the square root of the number of trading days in a year (typically 252)[5][7]. The basic steps are: calculate average return, find each return’s deviation from the mean, square these deviations, average them (variance), and take the square root (standard deviation)[1][3][8].
